# pymefpp_create_associations

This notebook is used to create the associations required to run mefpp


In [ ]:
from os import listdir
from os.path import isfile, join
import glob
from astropy.io import fits
import math
import operator
import copy
#import datetime
import os
from datetime import datetime
from pathlib import Path

In [ ]:
class Exposure:
    
    def __init__(self, filename):

        self.filename = filename
        with fits.open(filename) as hdu_list:
            primary_header = hdu_list[0].header
            self.dpr_catg = self.getKeyVal(primary_header, 'HIERARCH ESO DPR CATG')
            self.dpr_type = self.getKeyVal(primary_header, 'HIERARCH ESO DPR TYPE')
            self.obsid = self.getKeyVal(primary_header, 'HIERARCH ESO OBS ID')
            self.OBstart = self.getKeyVal(primary_header, 'HIERARCH ESO OBS START')
            self.dateobs = self.getKeyVal(primary_header, 'DATE-OBS')
            self.tplnexp = self.getKeyVal(primary_header, 'HIERARCH ESO TPL NEXP')
            self.tplexpno = self.getKeyVal(primary_header, 'HIERARCH ESO TPL EXPNO')
            self.filter = self.getKeyVal(primary_header, 'HIERARCH ESO INS FILT1 NAME')
            self.exptime = self.getKeyVal(primary_header, 'EXPTIME')

            if (self.dpr_catg == 'SCIENCE' and self.dpr_type == 'OBJECT') or \
                (self.dpr_catg == 'CALIB' and self.dpr_type == 'STD,ZEROPOINT'):
                self.ra = self.getKeyVal(primary_header, 'RA')
                self.dec = self.getKeyVal(primary_header, 'DEC')
                self.tel_alpha = self.getKeyVal(primary_header, 'HIERARCH ESO TEL TARG ALPHA')
                self.tel_delta = self.getKeyVal(primary_header, 'HIERARCH ESO TEL TARG DELTA')
                self.target = self.getKeyVal(primary_header, 'HIERARCH ESO OBS TARG NAME').replace(' ', '_')
        
                # we add the FLI
                yyyy = int(self.dateobs[0:4])
                mm = int(self.dateobs[5:7])
                dd = int(self.dateobs[8:10])
                fli = self.get_phase_on_day(yyyy,mm,dd)
                if fli <= 0.5:
                    fli = fli*2.0
                else:
                    fli = 2.0*(1-fli)
                self.fli = fli  

    def get_phase_on_day(self, year, month, day):
        """Returns a floating-point number from 0-1. where 0=new, 0.5=full, 1=new"""
        #Ephem stores its date numbers as floating points, which the following uses
        #to conveniently extract the percent time between one new moon and the next
        #This corresponds (somewhat roughly) to the phase of the moon.

        #Use Year, Month, Day as arguments
        date=ephem.Date(datetime.date(year,month,day))

        nnm = ephem.next_new_moon    (date)
        pnm = ephem.previous_new_moon(date)

        lunation=(date-pnm)/(nnm-pnm)

        #Note that there is a ephem.Moon().phase() command, but this returns the
        #percentage of the moon which is illuminated. This is not really what we want.

        return lunation

            
    def getKeyVal(self,primary_header, key):
        val = primary_header[key]
        return val


class ExposuresPartition:
    
    def __init__(self, filename, frametype):

        #print('ExposurePartition  filename: '+filename+'\nExposurePartition frametype: '+frametype, end='\r')
        
        self.base_path = os.path.dirname(filename)
        self.frametype = frametype

        self.filter = None
        if frametype != 'BIAS' or frametype != 'BPM':
            self.filter = self.getFilter(filename)
        self.framelist = filename
        self.frame = None
        
        # for the moment the BPM have no time stamp
        self.OBstart = None
        if frametype != 'BPM':
            self.OBstart = self.getOBstart(filename)
            
        # we construct the names according to file type
        self.framelist = None
        self.frame = None
        if frametype == 'BPM':
            self.framelist = None
            self.frame = self.base_path+'/OMEGACAM_BPM_01.fits'
        elif frametype == 'BIAS':
            self.framelist = self.base_path+'/bias.list'
            self.frame = self.base_path+'/OMEGACAM_MB_1x1.fits'
        elif frametype == 'FLAT,DOME':
            self.framelist = self.base_path+'/dome.list'
            self.frame = self.base_path+'/OMEGACAM_DOMEFF_'+self.filter+'_1x1.fits'
        elif frametype == 'FLAT,SKY':
            self.framelist = self.base_path+'/sky.list'
            self.frame = self.base_path+'/OMEGACAM_SKYFF_'+self.filter+'_1x1.fits'
        elif frametype == 'MFF':
            self.framelist = self.base_path+'/mff.list'
            self.frame = self.base_path+'/OMEGACAM_MFF_'+self.filter+'_1x1.fits'
        elif frametype == 'SCIENCE' or frametype == 'SKY':
            self.framelist = self.base_path+'/science.list'            
            self.frame = None


        self.bpmPartition = None
        self.bpmFrame = None
        self.bpm_have = False
        self.bpm_have_link = False

        self.biasPartition = None
        self.biasFrame = None
        self.bias_have = False
        self.bias_have_link = False

        self.domePartition = None
        self.domeFrame = None

        self.skyPartition = None
        self.SkyFrame = None

        self.mffPartition = None
        self.mffFrame = None
        
    def insertExposure(self, exposure):

        # we select the partition according to exposure dpr.type
        if 'BIAS' in exposure.dpr_type:
            key_str = 'bias/'+'%d'%exposure.obsid+'/'+exposure.OBstart
        elif 'FLAT,DOME' in exposure.dpr_type:
            key_str = 'dome/'+'%d'%exposure.obsid+'/'+exposure.filter+'/'+exposure.OBstart
        elif 'FLAT,SKY' in exposure.dpr_type:
            key_str = 'sky/'+'%d'%exposure.obsid+'/'+exposure.filter+'/'+exposure.OBstart
        elif ('CALIB' in exposure.dpr_catg) and ('STD,ZEROPOINT' in exposure.dpr_type):
            key_str = 'std/'+exposure.target + '_' + \
                    exposure.filter+'/'+\
                    exposure.OBstart
        elif ('SCIENCE' in exposure.dpr_catg) and ('OBJECT' in exposure.dpr_type):
            key_str = 'science/'+exposure.target+'_'+\
                    ("%010.3f"%exposure.tel_alpha).replace('.','p')+'_'+\
                    ("%09.1f"%exposure.tel_delta).replace('.','p').replace('-','m')+'/'+\
                    exposure.filter+'/'+\
                    exposure.OBstart
        else:
            print('dpr type: %s'%exposure.dpr_type, ' not classifiable.', end='\r')
            key_str = 'unknown/'+'%s'%exposure.dpr_type


        self.base_path = key_str
        self.exposureList.append(exposure)
        self.OBstart = exposure.OBstart
        self.filter = exposure.filter
        if key_str in self.partList:
            self.partList[key_str].append(exposure)
        else:
            self.partList[key_str] = [exposure]
            self.OBstartDict[key_str] = exposure.OBstart
            self.filterDict[key_str] = exposure.filter
        
    def getFilter(self,filename):
        fields = filename.split('/')
        return fields[-3]
        
    def getOBstart(self, filename):
        fields = filename.split('/')
        return fields[-2]

def timeDistance(time_str1, time_str2):
    FMT = '%Y-%m-%dT%H:%M:%S'
    return abs((datetime.strptime(time_str1, FMT) - datetime.strptime(time_str2, FMT)).total_seconds())

def getKeyVal(filename, key):
    with fits.open(filename) as hdu_list:
        primary_header = hdu_list[0].header
        val = primary_header[key]
        return val

class SciencePartition:
    
    def __init__(self, filename):

        self.base_path = os.path.dirname(filename)
        self.filter = self.getFilter(filename)
        self.OBstart = self.getOBstart(filename)
        self.bpmPartition = None
        self.biasPartition = None
        self.biasFrame = None
        self.domePartition = None
        self.domeFrame = None
        self.skyPartition = None
        self.SkyFrame = None
        self.mffPartition = None
        self.mffFrame = None
        
    def getFilter(self,filename):
        fields = filename.split('/')
        return fields[-3]
        
    def getOBstart(self, filename):
        fields = filename.split('/')
        return fields[-2]
    
class BiasPartition:
    
    def __init__(self, filename):

        self.base_path = os.path.dirname(filename)
        self.OBstart = self.getOBstart(filename)
        self.bpmPartition = None
        #self.biasPartition = None
        self.mffPartition = None
        
    def getOBstart(self, filename):
        fields = filename.split('/')
        return fields[-2]    

def timeDistance(time_str1, time_str2):
    FMT = '%Y-%m-%dT%H:%M:%S'
    return abs((datetime.strptime(time_str1, FMT) - datetime.strptime(time_str2, FMT)).total_seconds())

def getKeyVal(filename, key):
    with fits.open(filename) as hdu_list:
        primary_header = hdu_list[0].header
        val = primary_header[key]
        return val

In [ ]:
# we set the data reduction directories
raw_data_dir = '/home/lschmidt/fernando/fits/'
reduced_cal_dir = '/home/fselman/omegacam/fernando/reduced/calibs/'
reduced_science_dir = '/home/fselman/omegacam/fernando/reduced/science/'

In [ ]:
bpm_list = glob.glob(reduced_cal_dir+'**/bpm.list', recursive=True)
print('number of bpm partitions: ', len(bpm_list))
bpm_partitions = []
for bpm_file in bpm_list:
    bpm_partitions.append(BiasPartition(bpm_file))
    
for part in bpm_partitions:
    print(part.OBstart)

In [ ]:
bias_list = glob.glob(reduced_cal_dir+'**/bias.list', recursive=True)
print('number of bias partitions: ', len(bias_list))
bias_partitions = []
for bias_file in bias_list:
    bias_partitions.append(BiasPartition(bias_file))
    
#for part in bias_partitions:
#    print(part.OBstart)

In [ ]:
dome_list = glob.glob(reduced_cal_dir+'**/dome.list', recursive=True)
print('number of dome partitions: ', len(dome_list))
dome_partitions = []
for dome_file in dome_list:
    #print(dome_file)
    dome_partitions.append(SciencePartition(dome_file))
    
#for part in dome_partitions:
#    print(part.OBstart, part.filter)

In [ ]:
sky_list = glob.glob(reduced_cal_dir+'**/sky.list', recursive=True)
print('number of sky partitions: ', len(sky_list))
sky_partitions = []
for sky_file in sky_list:
    #print(sky_file)
    sky_partitions.append(SciencePartition(sky_file))
    
#for part in sky_partitions:
#    print(part.OBstart, part.filter)

In [ ]:
science_list = glob.glob(reduced_science_dir+'**/science.list', recursive=True)
print('number of science partitions: ', len(science_list))

sci_partitions = []
for science_file in science_list:
    sci_partitions.append(SciencePartition(science_file))
    
#for part in sci_partitions:
#    print(part.OBstart, part.filter)



In [ ]:
#We associate a bpm partition to each science frame
for sci_part in sci_partitions:
    sci_part.bpmPartition = bpm_partitions[0]
    #print((sci_part.bpmPartition).base_path)
    for i in range(1,33):
        bpm_file = (sci_part.bpmPartition).base_path+'/OMEGACAM_BPM_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, sci_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((sci_part.bpmPartition).base_path+'/toreduce_sci').touch()
        print(bpm_file, sci_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))


In [ ]:

# we associate a bias partition to each science partition

for sci_part in sci_partitions:
    # here we look for the closest in time bias partition
    sci_date = sci_part.OBstart

    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = timeDistance(sci_date, bias_date)
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            sci_part.biasPartition = bias_part
    
    # we find the sky closest in time and we link the mff to a file that
    # will be present there once it is created
    #
    skyBestDeltaT = 3600.*24.*30.0
    for sky_part in sky_partitions:
        if sci_part.filter != sky_part.filter:
            continue
        sky_date = sky_part.OBstart
        deltaT = timeDistance(sci_date, sky_date)

        if deltaT < skyBestDeltaT:
            skyBestDeltaT = deltaT
            sci_part.skyPartition = sky_part
            sci_part.mffPartition = sky_part

    
    print('science: %s %s'%(sci_part.filter, sci_part.OBstart,)+\
            '     bias: %s (%3.1f)'%(sci_part.biasPartition.OBstart,biasBestDeltaT/3600.0/24.,)+\
            ' sky: %s %s (%3.1f)'%(sci_part.skyPartition.filter, sci_part.skyPartition.OBstart, skyBestDeltaT/3600.0/24.,))

# at this point we have the biases and mff for each science frame. Now we link
for sci_part in sci_partitions:
    mbias = (sci_part.biasPartition).base_path+'/OMEGACAM_MB_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, sci_part.base_path+'/OMEGACAM_MB_1x1.fits')
        except:
            pass
        Path((sci_part.biasPartition).base_path+'/toreduce_sci').touch()
        print(mbias)
        
#    skyff = (sci_part.skyPartition).base_path+'/OMEGACAM_SKYFF_'+sci_part.filter+'_1x1.fits'
#    if not Path(skyff).is_file():
#        try:
#            os.symlink(skyff, sci_part.base_path+'/OMEGACAM_SKYFF_'+sci_part.filter+'_1x1.fits')
#        except:
#            pass
#        Path((sci_part.skyPartition).base_path+'/toreduce_sci').touch() 

    mff = (sci_part.mffPartition).base_path+'/OMEGACAM_MFF_'+sci_part.filter+'_1x1.fits'
    if not Path(mff).is_file():
        try:
            os.symlink(mff, sci_part.base_path+'/OMEGACAM_MFF_'+sci_part.filter+'_1x1.fits')
        except:
            pass
        Path((sci_part.mffPartition).base_path+'/toreduce_mff').touch()


In [ ]:
#We associate a bpm partition to each sky frame partition
for sky_part in sky_partitions:
    sky_part.bpmPartition = bpm_partitions[0]
    #print((sky_part.bpmPartition).base_path)
    for i in range(1,33):
        bpm_file = (sky_part.bpmPartition).base_path+'/OMEGACAM_BPM_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, sky_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((sky_part.bpmPartition).base_path+'/toreduce_sky').touch()
        print(bpm_file, sky_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))

In [ ]:
# here we associate dome and bias to each sky partition
for sky_part in sky_partitions:
    sky_date = sky_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = timeDistance(sky_date, bias_date)
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            sky_part.biasPartition = bias_part
    #print(biasBestDeltaT/3600.0/24.0)
    
    # we find the dome closest in time and we link the domeff to a file that
    # will be present there once it is created
    #
    domeBestDeltaT = 3600.*24.*30.0
    for dome_part in dome_partitions:
        if sky_part.filter == dome_part.filter:
            dome_date = dome_part.OBstart
            deltaT = timeDistance(sky_date, dome_date)
            #print(sci_date, bias_date, deltaT)
            if deltaT < domeBestDeltaT:
                domeBestDeltaT = deltaT
                sky_part.domePartition = dome_part
        else:
            pass
    #print(domeBestDeltaT/3600.0/24.0)
    
    print('sky: %s %s'%(sky_part.filter, sky_part.OBstart,)+\
            '     bias: %s (%4.2f)'%(sky_part.biasPartition.OBstart,biasBestDeltaT/3600.0/24.,)+\
            ' dome: %s %s (%4.2f)'%(sky_part.domePartition.filter, sky_part.domePartition.OBstart, domeBestDeltaT/3600.0/24.,))
    
# at this point we have the biases and domes for each sky ff frame. Now we link
for sky_part in sky_partitions:
    mbias = (sky_part.biasPartition).base_path+'/OMEGACAM_MB_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, sky_part.base_path+'/OMEGACAM_MB_1x1.fits')
        except:
            pass
        Path((sky_part.biasPartition).base_path+'/toreduce_sky').touch()
        #print(mbias)
        
    domeff = (sky_part.domePartition).base_path+'/OMEGACAM_DOMEFF_'+sky_part.filter+'_1x1.fits'
    if not Path(domeff).is_file():
        try:
            os.symlink(domeff, sky_part.base_path+'/OMEGACAM_DOMEFF_'+sky_part.filter+'_1x1.fits')
        except:
            pass
        Path((sky_part.domePartition).base_path+'/toreduce_sky').touch()
        #print(domeff)    

In [ ]:
#We associate a bpm partition to each dome ff frame
for dome_part in dome_partitions:
    dome_part.bpmPartition = bpm_partitions[0]
    #print((dome_part.bpmPartition).base_path)
    for i in range(1,33):
        bpm_file = (dome_part.bpmPartition).base_path+'/OMEGACAM_BPM_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, dome_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((dome_part.bpmPartition).base_path+'/toreduce_dome').touch()
        #print(bpm_file, dome_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))

    dome_date = dome_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = timeDistance(dome_date, bias_date)
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            dome_part.biasPartition = bias_part        
    
    mbias = (dome_part.biasPartition).base_path+'/OMEGACAM_MB_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, dome_part.base_path+'/OMEGACAM_MB_1x1.fits')
        except:
            pass
        Path((dome_part.biasPartition).base_path+'/toreduce_dome').touch()
        #print(mbias)

In [ ]:
# here we associate biases to each dome partition
for dome_part in dome_partitions:
    dome_date = dome_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = timeDistance(dome_date, bias_date)
        
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            dome_part.biasPartition = bias_part
            #print(dome_date, bias_date, biasBestDeltaT/3600.0/24.0)

    
    print('dome: %s %s'%(dome_part.filter, dome_part.OBstart,)+\
            '     bias: %s (%5.3f)'%((dome_part.biasPartition).OBstart, biasBestDeltaT/3600.0/24.,))
    
# at this point we have the biases  for each dome ff frame. Now we link
for dome_part in dome_partitions:
    mbias = (dome_part.biasPartition).base_path+'/OMEGACAM_MB_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, dome_part.base_path+'/OMEGACAM_MB_1x1.fits')
        except:
            pass
        Path((dome_part.biasPartition).base_path+'/toreduce_dome').touch()
        #print(mbias)
        

In [ ]:
##Finalle associate a bpm partition to each bias frame
for bias_part in bias_partitions:
    bias_part.bpmPartition = bpm_partitions[0]
    #print((bias_part.bpmPartition).base_path)
    for i in range(1,33):
        bpm_file = (bias_part.bpmPartition).base_path+'/OMEGACAM_BPM_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, bias_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((bias_part.bpmPartition).base_path+'/toreduce_bias').touch()
        #print(bpm_file, bias_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))